In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import utulek

utulek.platform.import_globals_notebook(globals())

base_module = utulek
IS_NOTEBOOK_KERNEL_CODE = True
NOTEBOOK_NAME = 2026_08_19-maildir_dedupe.ipynb
ASSET_PATH = /home/gilgamesh/main.syncthing/utulek/experiment/2026_08_19-maildir_dedupe.ipynb.asset/
Loaded `/home/gilgamesh/main.syncthing/utulek/experiment/.env`.
torch: ['NVIDIA GeForce GTX 1080 Ti']
tf: /device:GPU:0
jax: [CudaDevice(id=0)]


In [20]:
import hashlib
import re
from email import policy
from email.parser import BytesParser
from email.utils import getaddresses


def normalize_text(s):
	s = s.replace("\r\n", "\n").replace("\r", "\n")
	# Collapse whitespace, including the differences caused by wrapping.
	s = re.sub(r"\s+", " ", s)
	return s.strip()


def normalized_addresses(value):
	addresses = getaddresses([value or ""])
	return sorted(
		f"{name.strip().lower()} <{addr.strip().lower()}>"
		if name.strip() else addr.strip().lower()
		for name, addr in addresses)


def get_text_parts(msg):
	parts = []

	if msg.is_multipart():
		for part in msg.walk():
			if part.get_content_type(
			) == "text/plain" or part.get_content_type(
			) == "text/html":
				try:
					parts.append(part.get_content())
				except Exception:
					payload = part.get_payload(decode=True)
					if payload:
						parts.append(
							payload.decode(
								part.get_content_charset() or "utf-8",
								errors="replace",
							))
			else:
				if part.get_filename():
					parts.append(part.get_filename())
	elif msg.get_content_type() == "text/plain":
		try:
			parts.append(msg.get_content())
		except Exception:
			payload = msg.get_payload(decode=True)
			if payload:
				parts.append(
					payload.decode(
						msg.get_content_charset() or "utf-8",
						errors="replace",
					))
	elif msg.get_content_type() == "text/html":
		try:
			parts.append(msg.get_content())
		except Exception:
			payload = msg.get_payload(decode=True)
			if payload:
				parts.append(
					payload.decode(
						msg.get_content_charset() or "utf-8",
						errors="replace",
					))

	return parts


def fingerprint(raw_message):
	msg = BytesParser(
		policy=policy.default).parsebytes(raw_message)

	pieces = []

	for header in ("From", "To", "Cc", "Subject"):
		if header.lower() == "subject":
			value = normalize_text(msg.get(header, "")).casefold()
			pieces.append(("subject", value))
		else:
			pieces.append(
				(
					header.lower(),
					normalized_addresses(msg.get(header, ""))))

	body = "\n".join(
		normalize_text(x) for x in get_text_parts(msg))
	pieces.append(("body", body))

	canonical = repr(pieces).encode("utf-8")
	return hashlib.sha256(canonical).hexdigest()

In [58]:
from multiprocessing import Pool

maildir = "/home/gilgamesh/main.syncthing/monochrome/archive.syncthing/module/email/archive.maildir/cur/"


def process_file(file):
	with open(maildir + file, "rb") as f:
		raw = f.read()
	fp = fingerprint(raw)
	return (file, fp)


with Pool(32) as pool:
	file_fp = list(
		tqdm(
			pool.imap_unordered(
				process_file, os.listdir(maildir))))

/usr/local/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


0it [00:00, ?it/s]

In [59]:
print(len(file_fp))
groups_all = {}
for ffp in file_fp:
	if ffp[1] not in groups_all:
		groups_all[ffp[1]] = []
	groups_all[ffp[1]].append(ffp[0])
print(len(groups_all))
groups = []
for fp in groups_all:
	if len(groups_all[fp]) > 1:
		groups.append(groups_all[fp])
print(len(groups))

18578
18196
106


In [60]:
from datetime import timedelta, timezone, datetime
from email import policy
from email.parser import BytesParser
from email.utils import parsedate_to_datetime


def get_date(filename):
	try:
		with open(filename, "rb") as f:
			msg = BytesParser(policy=policy.default).parse(f)
			value = msg.get("Date")
			if not value:
				return None

			dt = parsedate_to_datetime(value)

			if dt.tzinfo is None:
				# Treat offset-naive dates as UTC.
				return dt

			# Convert offset-aware dates to naive UTC.
			return dt.astimezone(
				timezone.utc).replace(tzinfo=None)
	except (OSError, ValueError):
		return None

In [61]:
groups_good = []
groups_bad = []

for group in groups:
	dates = [
		get_date(maildir + filename) for filename in group
	]

	if all(date is not None for date in dates):
		if max(dates) - min(dates) <= timedelta(days=1):
			groups_good.append(group)
			continue

	groups_bad.append(group)
	print(dates, group)

print(len(groups_good), len(groups_bad))

[datetime.datetime(2026, 6, 2, 13, 6, 46), datetime.datetime(2026, 6, 4, 14, 6, 46)] ['1786954592.R6296934009546395543.gilgamesh-51;2,S', '1786954592.R2327877951528357142.gilgamesh-51;2,S']
[datetime.datetime(2026, 1, 17, 20, 58, 39), datetime.datetime(2026, 5, 16, 17, 58, 51), datetime.datetime(2026, 3, 18, 22, 36, 10), datetime.datetime(2026, 2, 18, 18, 27, 42), datetime.datetime(2026, 4, 16, 16, 53, 3)] ['1786954660.R6619839306515217246.gilgamesh-51;2,S', '1786954595.R9268458772210918105.gilgamesh-51;2,S', '1786954609.R8905240683033731149.gilgamesh-51;2,S', '1786954615.R8261721490119663760.gilgamesh-51;2,S', '1786954604.R12151175649015140380.gilgamesh-51;2,S']
[datetime.datetime(2025, 2, 4, 5, 59, 46), datetime.datetime(2025, 1, 4, 13, 49, 38), datetime.datetime(2025, 1, 3, 16, 24, 26)] ['1786954919.R6025302024470548572.gilgamesh-51;2,S', '1786954920.R10188772200821888147.gilgamesh-51;2,S', '1786954920.R17941219684797987352.gilgamesh-51;2,S']
[datetime.datetime(2025, 11, 2, 9, 21, 5

In [62]:
groups = groups_good

In [63]:
groups_ordered = []


def has_return_path(filename):
	with open(filename, "rb") as f:
		msg = BytesParser(policy=policy.default).parse(f)

	return msg.get("Return-Path") is not None


for group in groups:
	matches = [
		filename for filename in group
		if has_return_path(maildir + filename)
	]

	if len(matches) != 1:
		dates = sorted(
			[
				(get_date(maildir + filename), filename)
				for filename in group
			])
		groups_ordered.append([i[1] for i in dates])
	else:
		for filename in group:
			if filename not in matches:
				matches.append(filename)
		groups_ordered.append(matches)

print(groups_ordered)

[['1786954585.R4204217906665784351.gilgamesh-51;2,S', '1786954585.R1441779395736274962.gilgamesh-51;2,S'], ['1786954910.R11959935929137330411.gilgamesh-51;2,S', '1786954910.R8927727377444710956.gilgamesh-51;2,S', '1786954910.R8642331237759326595.gilgamesh-51;2,S', '1786954910.R6495483067193829951.gilgamesh-51;2,S'], ['1786954670.R13081068708818977752.gilgamesh-51;2,S', '1786954670.R809314343655930887.gilgamesh-51;2,S'], ['1786954914.R12428386130025618672.gilgamesh-51;2,S', '1786954914.R3170070877077330294.gilgamesh-51;2,S', '1786954914.R6236873124392418287.gilgamesh-51;2,S', '1786954914.R6438806701728181945.gilgamesh-51;2,S', '1786954914.R2821620254902252377.gilgamesh-51;2,S', '1786954914.R16872904847157842166.gilgamesh-51;2,S', '1786954913.R197067829202029424.gilgamesh-51;2,S', '1786954913.R6575739737078542732.gilgamesh-51;2,S', '1786954913.R7852564422408790972.gilgamesh-51;2,S', '1786954913.R5401522199266814335.gilgamesh-51;2,S'], ['1786954585.R14940333845557745333.gilgamesh-51;2,S',

In [64]:
for group in groups_ordered:
	for filename in group[1:]:
		pathlib.Path(maildir + filename).unlink()